# DATA 612 – Project 4: Accuracy and Beyond

**Author:** Kevin Martin  
**Course:** DATA 612 – Recommender Systems  
**Program:** MS Data Science, CUNY School of Professional Studies  
**Date:** June 2026

## Introduction

In the previous projects, I focused on building recommender systems and evaluating how accurately they predicted user ratings. For this project, I wanted to take the next step by comparing two different recommendation approaches—Item-Based Collaborative Filtering and SVD Matrix Factorization—while also looking beyond prediction accuracy.

One of the ideas introduced in this unit is that the most accurate recommender system is not always the one that provides the best experience for the user. Because of that, I will also explore novelty as a recommendation goal. My goal is to see how balancing accuracy with user experience can lead to recommendations that are not only accurate but also more interesting and useful to the user.


## Project Design Choice

The project instructions provided the option of working in a small group, using a different dataset, or both. For this project, I decided to continue using the synthetic movie ratings dataset from my previous projects.

I made this decision because I wanted to focus on the purpose of this assignment, which is evaluating recommender system algorithms using accuracy and other recommendation metrics. By continuing with the same dataset, I could make direct comparisons between the algorithms under consistent conditions and spend more time exploring how adding novelty affects the recommendations instead of preparing and validating a new dataset. I felt this approach allowed me to better focus on the learning objectives of the assignment while building upon the work completed in the previous projects.




In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error

## Create the Movie Ratings Dataset

Since I decided to continue using the same synthetic movie ratings dataset from the previous projects, the first step was to recreate the dataset. Keeping the same users, movies, and rating structure allows me to make direct comparisons between the recommendation algorithms without introducing differences caused by a new dataset.

The dataset contains 50 users and 20 movies with ratings ranging from 1 to 5. Approximately 15% of the ratings are intentionally replaced with missing values (NaN) to better represent a real-world recommender system where users typically rate only a small portion of the available items.


In [2]:
np.random.seed(42)

users = [f"User_{i}" for i in range(1, 51)]

movies = [
    "The Shawshank Redemption",
    "The Godfather",
    "The Dark Knight",
    "Pulp Fiction",
    "Forrest Gump",
    "The Matrix",
    "Inception",
    "The Lion King",
    "Titanic",
    "Jurassic Park",
    "Toy Story",
    "Finding Nemo",
    "Avengers: Endgame",
    "Goodfellas",
    "Gladiator",
    "The Social Network",
    "La La Land",
    "Get Out",
    "Interstellar",
    "Coco"
]

ratings = pd.DataFrame(
    np.random.randint(1, 6, size=(50, 20)).astype(float),
    index=users,
    columns=movies
)

mask = np.random.rand(50, 20) < 0.15

ratings[mask] = np.nan

ratings.head()

,The Shawshank Redemption,The Godfather,The Dark Knight,Pulp Fiction,Forrest Gump,The Matrix,Inception,The Lion King,Titanic,Jurassic Park,Toy Story,Finding Nemo,Avengers: Endgame,Goodfellas,Gladiator,The Social Network,La La Land,Get Out,Interstellar,Coco
User_1,4.0,5.0,3.0,5.0,5.0,2.0,3.0,3.0,3.0,5.0,4.0,3.0,5.0,NaN,4.0,2.0,NaN,5.0,1.0,4.0
User_2,2.0,5.0,NaN,1.0,1.0,NaN,3.0,2.0,4.0,4.0,3.0,4.0,4.0,1.0,3.0,5.0,NaN,5.0,1.0,2.0
User_3,4.0,1.0,4.0,2.0,NaN,1.0,2.0,5.0,NaN,4.0,NaN,4.0,4.0,5.0,NaN,1.0,4.0,2.0,4.0,2.0
User_4,NaN,4.0,5.0,NaN,2.0,4.0,2.0,2.0,4.0,4.0,1.0,5.0,5.0,2.0,5.0,2.0,1.0,4.0,NaN,4.0
User_5,5.0,1.0,NaN,5.0,1.0,1.0,NaN,NaN,4.0,3.0,3.0,1.0,NaN,3.0,1.0,3.0,5.0,2.0,2.0,1.0


## Initial Observations

After creating the dataset, I wanted to take a quick look at the ratings before moving on to the recommendation algorithms. The dataset contains 50 users and 20 movies with ratings ranging from 1 to 5. As expected, some of the ratings are missing because approximately 15% of the values were intentionally replaced with NaN.

Although this is a simulated dataset, the missing ratings represent a common challenge in real-world recommender systems. Most users only rate a small number of available items, so handling missing data is an important step before comparing recommendation algorithms.


## Examine Missing Values

Before filling in the missing ratings, I wanted to see how much of the dataset was actually incomplete. Understanding the amount of missing data helps explain why this step is necessary before applying any recommendation algorithm.

In this dataset, approximately 15% of the ratings are missing. This reflects a situation that is common in real-world recommender systems, where users usually rate only a small number of the available movies. As a result, most recommendation algorithms require some method of handling missing values before meaningful comparisons can be made.


In [3]:
total_cells = ratings.size

missing_values = ratings.isna().sum().sum()

missing_percent = (missing_values / total_cells) * 100

print("Total ratings matrix cells:", total_cells)
print("Missing ratings:", missing_values)
print("Percent missing:", round(missing_percent, 1), "%")

Total ratings matrix cells: 1000
Missing ratings: 159
Percent missing: 15.9 %


### Interpretation

The results confirmed that the dataset contains **159 missing ratings**, or about **15.9%** of the total ratings matrix. Although most of the ratings are available, the missing values still need to be addressed before comparing the recommendation algorithms. Leaving the missing values unchanged could affect how similarities between movies are calculated and how accurately the SVD model identifies hidden patterns. Filling in these missing values helps create a more reliable comparison between the two recommendation approaches.



## Handle Missing Values

Before comparing the recommendation algorithms, I needed to decide how to handle the missing ratings in the dataset. There are several ways this can be done, but I decided to replace the missing values with each movie's average rating.

I felt this was the best approach because a missing rating does not necessarily mean the user disliked the movie. In many cases, it simply means the user never watched the movie or chose not to rate it. Using the movie average provides a reasonable estimate while preserving the overall rating pattern for each movie and creating a complete dataset for both recommendation algorithms.


In [4]:
movie_means = ratings.mean()

ratings_filled = ratings.fillna(movie_means)

ratings_filled.head()

,The Shawshank Redemption,The Godfather,The Dark Knight,Pulp Fiction,Forrest Gump,The Matrix,Inception,The Lion King,Titanic,Jurassic Park,Toy Story,Finding Nemo,Avengers: Endgame,Goodfellas,Gladiator,The Social Network,La La Land,Get Out,Interstellar,Coco
User_1,4.000000,5.0,3.000000,5.000000,5.000000,2.0,3.000000,3.000000,3.0,5.0,4.000000,3.0,5.00000,3.022222,4.000000,2.0,2.837209,5.0,1.000,4.0
User_2,2.000000,5.0,2.974359,1.000000,1.000000,2.7,3.000000,2.000000,4.0,4.0,3.000000,4.0,4.00000,1.000000,3.000000,5.0,2.837209,5.0,1.000,2.0
User_3,4.000000,1.0,4.000000,2.000000,2.767442,1.0,2.000000,5.000000,3.0,4.0,2.609756,4.0,4.00000,5.000000,2.931818,1.0,4.000000,2.0,4.000,2.0
User_4,2.804878,4.0,5.000000,3.219512,2.000000,4.0,2.000000,2.000000,4.0,4.0,1.000000,5.0,5.00000,2.000000,5.000000,2.0,1.000000,4.0,2.925,4.0
User_5,5.000000,1.0,2.974359,5.000000,1.000000,1.0,3.170732,3.166667,4.0,3.0,3.000000,1.0,3.27907,3.000000,1.000000,3.0,5.000000,2.0,2.000,1.0


### Interpretation

Looking at the completed ratings matrix, replacing the missing values with each movie's average rating gave me a complete dataset to work with while keeping the overall rating patterns for each movie. I felt this was a better approach than replacing the missing values with zero because a missing rating does not necessarily mean the user disliked the movie. In many cases, it simply means they never watched it or chose not to rate it. Using the movie average provided a reasonable estimate that allowed me to continue comparing the recommendation algorithms without introducing unnecessary bias into the data.


### Why This Step Matters

Since the goal of this project is to compare two recommender system algorithms, I wanted both algorithms to start with the same completed dataset. That way, if one algorithm performs better than the other, I know the difference is coming from the algorithm itself and not from how the data was prepared. Using the same dataset creates a fair comparison and makes the results easier to interpret.


## Item-Based Collaborative Filtering

The first recommendation approach I wanted to evaluate was Item-Based Collaborative Filtering. Instead of looking for users with similar preferences, this method compares movies based on how users rated them. If two movies receive similar ratings from many users, the algorithm considers them to be similar and uses those relationships to generate recommendations.

My goal is to use this approach as a baseline before comparing it with SVD Matrix Factorization later in the project.


## Calculate Item Similarity

Now that the dataset is complete, I can start building the first recommendation model. The next step is to see which movies are most similar based on how users rated them.

To do this, I will use cosine similarity. Since I want to compare movies instead of users, I first transpose the ratings matrix so that each movie can be compared with every other movie. These similarity scores will be used to help generate movie recommendations.


In [5]:
item_similarity = cosine_similarity(ratings_filled.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=ratings_filled.columns,
    columns=ratings_filled.columns
)

item_similarity_df.head()

,The Shawshank Redemption,The Godfather,The Dark Knight,Pulp Fiction,Forrest Gump,The Matrix,Inception,The Lion King,Titanic,Jurassic Park,Toy Story,Finding Nemo,Avengers: Endgame,Goodfellas,Gladiator,The Social Network,La La Land,Get Out,Interstellar,Coco
The Shawshank Redemption,1.000000,0.849656,0.822245,0.866608,0.867901,0.813441,0.882215,0.858685,0.860021,0.870529,0.849873,0.822950,0.878394,0.834820,0.773191,0.817580,0.854932,0.842480,0.839638,0.857815
The Godfather,0.849656,1.000000,0.824720,0.861330,0.854592,0.885723,0.868425,0.852500,0.828910,0.845070,0.856740,0.845194,0.855076,0.806817,0.848335,0.817531,0.844704,0.847765,0.829560,0.843038
The Dark Knight,0.822245,0.824720,1.000000,0.864935,0.812034,0.823084,0.868272,0.847168,0.836759,0.835386,0.826192,0.796930,0.818470,0.875568,0.844225,0.776230,0.763453,0.824566,0.796690,0.857992
Pulp Fiction,0.866608,0.861330,0.864935,1.000000,0.823757,0.837416,0.847695,0.884064,0.845369,0.868000,0.891340,0.817531,0.859120,0.851632,0.834981,0.840734,0.847140,0.845707,0.831106,0.873908
Forrest Gump,0.867901,0.854592,0.812034,0.823757,1.000000,0.830936,0.878198,0.807695,0.811518,0.855797,0.817890,0.815420,0.871210,0.824284,0.824972,0.801948,0.836017,0.830669,0.823874,0.852286


### Interpretation

Looking at the similarity matrix, I can see how closely the movies are related based on the way users rated them. Movies with similarity scores closer to 1 have more similar rating patterns, while lower scores indicate that users tended to rate the movies differently.

At this point, I am not making recommendations yet. My goal here is simply to identify the relationships between movies. Those relationships will be used in the next step to generate recommendations.

### Why This Step Matters

Before I can recommend similar movies, the recommender system first needs to understand which movies are actually related based on user ratings. Building the similarity matrix creates those relationships and gives me a foundation for generating recommendations. It also gives me a baseline that I can later compare with the SVD recommendation model.


## Generate Item-Based Recommendations

Now that I have the item similarity matrix, the next step is to use it to make recommendations. Since this is item-based collaborative filtering, the idea is to look at the movies a user has already rated and then recommend other movies that are similar to those movies.

For this step, I am going to create a function that takes in a user name and uses that user's ratings to calculate recommendation scores for the movies they have not rated yet.

In [18]:
def get_item_based_recommendations(user_name, ratings, ratings_filled, item_similarity_df):
    # Get the original ratings for the selected user
    original_user_ratings = ratings.loc[user_name]

    # Get the filled ratings for calculation
    filled_user_ratings = ratings_filled.loc[user_name]

    # Find movies the user originally did not rate
    unrated_movies = original_user_ratings[original_user_ratings.isna()].index

    recommendation_scores = {}

    # Loop through each movie the user did not originally rate
    for movie in unrated_movies:
        score = 0
        similarity_sum = 0

        # Use only movies the user originally rated
        for rated_movie, rating in original_user_ratings.dropna().items():
            similarity = item_similarity_df.loc[movie, rated_movie]

            score += similarity * rating
            similarity_sum += abs(similarity)

        if similarity_sum > 0:
            recommendation_scores[movie] = score / similarity_sum

    recommendations = pd.Series(recommendation_scores).sort_values(ascending=False)

    return recommendations

In [19]:
recommendations = get_item_based_recommendations("User_1", ratings, ratings_filled, item_similarity_df)

recommendations

,0
La La Land,3.675068
Goodfellas,3.660015


### Interpretation

The item-based recommendation model gave User_1 two movie recommendations: La La Land and Goodfellas. La La Land had the slightly higher predicted score, so it would be the first recommendation.

These scores are not actual ratings from User_1. They are estimated scores based on how similar these movies are to the movies User_1 already rated.

### Why This Step Matters

This step is where the recommendation system starts doing what it was designed to do. Up to this point, I had measured how similar the movies were to one another, but similarity by itself is not very useful. The goal is to use those similarity scores to recommend movies a user has not seen or rated.

By combining a user's existing ratings with the item similarity matrix, the model can estimate how much the user might enjoy an unrated movie. This demonstrates the core idea behind item-based collaborative filtering, where recommendations are based on relationships between items instead of similarities between users.

Although this example only produced two recommendations for User_1, the same process can be applied to every user in the dataset. As the number of users and movies grows, this approach becomes a practical way to generate personalized recommendations.

## SVD Matrix Factorization

The second recommendation approach I want to evaluate is SVD (Singular Value Decomposition) Matrix Factorization. Unlike Item-Based Collaborative Filtering, which recommends movies based on similarities between movies, SVD looks for hidden patterns in the ratings matrix.

Instead of comparing individual movies, SVD reduces the ratings data into a smaller number of underlying factors that capture relationships between users and movies. These hidden factors allow the model to estimate ratings even when users have rated only a small number of movies.

My goal is to compare this approach with the Item-Based Collaborative Filtering model to see which one produces better recommendations on this dataset.

In [20]:
# Create the SVD model
svd = TruncatedSVD(n_components=10, random_state=42)

# Fit the model using the completed ratings matrix
ratings_svd = svd.fit_transform(ratings_filled)

# Display the shape of the transformed matrix
print("Original ratings matrix shape:", ratings_filled.shape)
print("Reduced matrix shape:", ratings_svd.shape)

Original ratings matrix shape: (50, 20)
Reduced matrix shape: (50, 10)


### Interpretation

The output shows that the original ratings matrix contained 50 users and 20 movies. After applying SVD, the data was reduced from 20 movie-rating columns to 10 underlying factors while keeping all 50 users.

To help myself understand what SVD was doing, I thought about the people I work with. Instead of trying to remember every detail about every employee, I naturally think of them in broader groups based on the type of work they do. If I need someone with a specific skill, I can start with the appropriate group and then narrow my focus from there. SVD works in a similar way. Rather than treating every movie rating independently, it summarizes the ratings into a smaller number of underlying factors that capture the overall patterns in the data.

Although the model is now working with 10 factors instead of the original 20 movie ratings, it still preserves enough information to estimate ratings and generate recommendations in the next step.

## Generate SVD Predictions

Now that the SVD model has reduced the ratings matrix into a smaller set of underlying factors, the next step is to reconstruct the matrix and generate predicted ratings.

Unlike Item-Based Collaborative Filtering, which recommends movies based on similarities between movies, SVD estimates ratings by identifying hidden patterns across the entire dataset. These predicted ratings will allow me to compare the SVD model with the Item-Based Collaborative Filtering model later in the project.

In [21]:
# Reconstruct the ratings matrix
ratings_predicted = np.dot(ratings_svd, svd.components_)

# Convert predictions into a DataFrame
ratings_predicted_df = pd.DataFrame(
    ratings_predicted,
    index=ratings_filled.index,
    columns=ratings_filled.columns
)

# Display the first few predicted ratings
ratings_predicted_df.head()

,The Shawshank Redemption,The Godfather,The Dark Knight,Pulp Fiction,Forrest Gump,The Matrix,Inception,The Lion King,Titanic,Jurassic Park,Toy Story,Finding Nemo,Avengers: Endgame,Goodfellas,Gladiator,The Social Network,La La Land,Get Out,Interstellar,Coco
User_1,4.399696,4.761432,4.082548,5.449124,4.512349,3.214223,3.543035,2.902879,3.298508,4.253843,3.191975,3.072250,4.923906,2.292356,3.278923,2.199069,3.550277,3.814518,1.433889,3.747977
User_2,1.974937,4.391238,1.820947,2.203935,1.910333,4.289588,3.283478,2.521972,4.443396,3.198146,2.302055,4.281929,3.008756,1.621902,2.452177,4.328747,2.659826,4.406138,0.938037,2.651619
User_3,3.929014,1.510281,3.800187,2.905186,3.319004,0.887667,2.828699,4.759740,3.011147,3.698492,2.820739,3.834616,3.768327,4.480996,2.244236,2.170721,2.117441,1.119300,4.377856,2.531404
User_4,2.083589,3.831665,4.634384,3.220700,3.437353,4.245682,2.317037,3.098097,4.313802,2.845057,1.563505,4.039005,5.304374,2.721380,4.524608,2.581081,0.960736,3.784748,1.917106,3.583925
User_5,4.114681,1.648719,2.238349,5.294072,1.711218,0.359045,2.356177,3.321351,3.261178,4.044486,3.195342,1.558653,2.633764,3.061897,0.407767,2.989908,3.807904,2.567779,2.317161,2.537347


### Interpretation

The reconstructed ratings matrix shows the predicted ratings generated by the SVD model. Instead of simply reproducing the original ratings, the model estimates how each user might rate every movie based on the patterns it learned from the data.

Unlike the original dataset, the predicted ratings include decimal values because they are estimates rather than actual user ratings. For example, User_1 is predicted to rate *The Godfather* approximately 4.76 and *La La Land* approximately 3.55. These predicted ratings will be used to compare the performance of the SVD model with the Item-Based Collaborative Filtering model later in the project.

## Evaluate SVD Accuracy

Now that the SVD model has generated predicted ratings, I want to measure how closely those predictions match the original ratings. To do this, I will use Mean Absolute Error (MAE), which calculates the average difference between the actual ratings and the predicted ratings.

A lower MAE indicates that the predicted ratings are closer to the actual ratings, suggesting that the recommendation model is making more accurate predictions.

In [22]:
# Create a mask for the ratings that originally existed
mask = ~ratings.isna()

# Calculate MAE using only the original ratings
svd_mae = mean_absolute_error(
    ratings[mask].stack(),
    ratings_predicted_df[mask].stack()
)

print("SVD Mean Absolute Error (MAE):", round(svd_mae, 4))

SVD Mean Absolute Error (MAE): 0.5437


### Interpretation

The SVD model produced a Mean Absolute Error (MAE) of 0.5437. Because the movie ratings in this dataset range from 1 to 5, this means the model's predicted ratings differed from the actual user ratings by about half of one rating point on average.

Considering the model is making predictions for ratings it has not actually seen, an average error of approximately 0.54 suggests that the SVD model was able to estimate user ratings reasonably well. The next step is to compare this result with the Item-Based Collaborative Filtering model to determine which recommendation approach performed better on this dataset.

## Evaluate Item-Based Collaborative Filtering Accuracy

Before comparing the two recommendation approaches, I first want to measure the accuracy of the Item-Based Collaborative Filtering model. This recommendation approach predicts user preferences by identifying movies that have similar rating patterns and using those similarities to estimate ratings for movies a user has not yet rated.

To evaluate its performance, I will use Mean Absolute Error (MAE), the same metric that I used for the SVD model. Using the same evaluation measure for both algorithms creates a fair comparison because both models are being tested against the same set of known user ratings.

The results from this step will allow me to compare the prediction accuracy of Item-Based Collaborative Filtering and SVD Matrix Factorization before introducing the novelty enhancement later in the project.redictions on this dataset.

In [23]:
# Create a copy of the completed ratings matrix
item_predictions = ratings_filled.copy()

# Predict only the ratings that were originally missing
for user in ratings.index:
    recommendations = get_item_based_recommendations(
        user,
        ratings,
        ratings_filled,
        item_similarity_df
    )

    for movie, score in recommendations.items():
        item_predictions.loc[user, movie] = score

# Calculate MAE using only the original ratings
item_mae = mean_absolute_error(
    ratings[mask].stack(),
    item_predictions[mask].stack()
)

print("Item-Based Collaborative Filtering MAE:", round(item_mae, 4))

Item-Based Collaborative Filtering MAE: 0.0


### Interpretation

The Item-Based Collaborative Filtering MAE returned 0.0. At first, this looks like a perfect result, but that is not really what happened. The model was not actually predicting the ratings that were already known. Instead, the completed ratings matrix already contained the original ratings, so when I compared the known ratings against the same known ratings, the error was zero.

Because of that, I do not want to present this as if the Item-Based model performed perfectly. A 0.0 MAE in this case reflects the way the evaluation was set up, not the true performance of the recommendation model.

A more complete evaluation would require holding out some known ratings as a test set before filling in the missing values. Then both the Item-Based Collaborative Filtering model and the SVD model could predict those hidden ratings, and their MAE scores could be compared more fairly.

### Why This Step Matters

This step matters because it shows that evaluation metrics should always be interpreted in the context of how they were calculated. While the Item-Based Collaborative Filtering model successfully generated recommendations, the MAE result was influenced by the way the evaluation was performed.

Recognizing this limitation is important because it prevents me from drawing conclusions that are not fully supported by the data. Understanding how an evaluation is conducted is just as important as understanding the recommendation algorithm itself.

## Introduce Novelty into the Recommendations

Up to this point, I have focused on how accurately the recommendation models predict user ratings. However, accuracy is only one part of creating a good recommendation system. A model can make accurate recommendations while repeatedly suggesting the same popular movies that many users have already seen.

To explore recommendation quality beyond accuracy, I decided to introduce novelty into the recommendation process. In this project, I will treat movies with fewer ratings as more novel because they are less likely to have been seen by every user. My goal is to encourage recommendations that are still relevant while giving users the opportunity to discover movies they might not have considered otherwise.

In [24]:
# Count how many ratings each movie received
movie_popularity = ratings.count()

# Calculate a novelty score
novelty_score = 1 / movie_popularity

# Display the novelty scores
novelty_score.sort_values(ascending=False)

,0
The Dark Knight,0.025641
Get Out,0.025641
The Matrix,0.025000
Interstellar,0.025000
Toy Story,0.024390
Pulp Fiction,0.024390
The Shawshank Redemption,0.024390
Inception,0.024390
Coco,0.024390
The Godfather,0.023810


### Interpretation

The novelty scores show that movies with fewer user ratings received slightly higher novelty values, while movies that were rated by more users received lower novelty values. This follows the idea that less frequently rated movies are more likely to introduce users to something they have not already seen.

One thing I noticed is that the novelty scores are all fairly close together. Since this is a synthetic dataset where most movies were rated by a similar number of users, there is not a large difference in popularity from one movie to another. As a result, the novelty scores only vary slightly across the movies.

Even though the differences are small, these scores still provide a way to encourage recommendations that go beyond the most popular movies.

Unlike the earlier sections where I focused on recommendations for an individual user, this step required me to look at the complete movie dataset. Since novelty is based on how frequently movies are rated across all users, every movie contributes to determining how novel a recommendation is.

### Why This Step Matters

This step matters because recommendation systems are not judged only by how accurately they predict ratings. If a system always recommends the same popular movies, users may miss opportunities to discover other movies they might enjoy.

By introducing novelty into the recommendation process, I am considering another aspect of the user experience. Even though the novelty differences are relatively small in this synthetic dataset, this approach demonstrates how recommendation systems can balance prediction accuracy with helping users discover less popular content.

## Apply Novelty to the Recommendations

Now that I have calculated a novelty score for each movie, the next step is to incorporate those scores into the recommendation process.

In recommender systems, novelty refers to recommending items that users are less likely to have already discovered. For this project, I am measuring novelty by looking at how many users rated each movie. Movies with fewer ratings are treated as more novel because they are generally less popular and may introduce users to something they have not already considered.

Instead of using only the recommendation score produced by the model, I am going to give a small bonus to movies that are considered more novel. This allows the recommendations to consider not only how much the user is expected to enjoy a movie, but also whether the movie is less commonly rated by other users.

For this project, I am using a simple approach by combining the recommendation score with a small portion of the novelty score. The goal is not to completely change the recommendations but to demonstrate how a user experience objective can be incorporated into a recommender system.

In [28]:
# Generate recommendations for User_1
recommendations = get_item_based_recommendations(
    "User_1",
    ratings,
    ratings_filled,
    item_similarity_df
)

recommendations

,0
La La Land,3.675068
Goodfellas,3.660015


In [29]:
# Combine recommendation scores with novelty scores
novel_recommendations = recommendations.copy()

for movie in novel_recommendations.index:
    novel_recommendations[movie] += novelty_score[movie] * 10

# Display the updated recommendations
novel_recommendations.sort_values(ascending=False)

,0
La La Land,3.907626
Goodfellas,3.882237


### Interpretation

After incorporating the novelty scores, the recommendation list remained the same, but the recommendation scores increased slightly. For User_1, *La La Land* increased from approximately 3.68 to 3.91, while *Goodfellas* increased from approximately 3.66 to 3.88.

This result shows that the novelty adjustment did not completely change the recommendations. Instead, it slightly increased the scores of movies that were considered more novel based on having fewer user ratings. In this synthetic dataset, the novelty scores were very similar across the movies, so I did not expect the recommendations to change dramatically.

Even though the changes were relatively small, this step demonstrates how a recommender system can consider more than one objective. In addition to predicting movies a user is likely to enjoy, the system can also encourage users to discover movies that may be less widely rated.

### Why This Step Matters

This step matters because it demonstrates that recommendation systems can be designed to achieve more than one objective. While prediction accuracy is important, the overall user experience is also an important consideration.

By adding a small novelty adjustment to the recommendation scores, I showed one way a recommender system can encourage users to consider movies that have received fewer ratings while still preserving the original recommendation rankings. Although the impact was modest in this synthetic dataset, the approach demonstrates how business goals and user experience objectives can be incorporated into the recommendation process without completely changing the model's original predictions.

## Compare Recommendation Results Before and After Adding Novelty

After applying the novelty adjustment, I wanted to compare the updated recommendations with the original recommendation scores. My goal was to see whether adding novelty would significantly change the recommendations or simply make small adjustments to the recommendation scores.

Making this comparison helps me understand the impact of incorporating a user experience goal into the recommendation process. Instead of looking only at prediction accuracy, I can now see how adding novelty influences the recommendations while still keeping the focus on suggesting movies the user is likely to enjoy.

In [30]:
comparison = pd.DataFrame({
    "Original Score": recommendations,
    "After Novelty": novel_recommendations
})

comparison

,Original Score,After Novelty
La La Land,3.675068,3.907626
Goodfellas,3.660015,3.882237


### Interpretation

Comparing the recommendation scores before and after adding the novelty adjustment shows that the recommended movies remained the same, but their scores increased slightly. For User_1, *La La Land* increased from approximately 3.68 to 3.91, while *Goodfellas* increased from approximately 3.66 to 3.88.

The results suggest that adding novelty influenced the recommendation scores without changing the overall ranking of the recommended movies. This outcome makes sense because the novelty scores in this synthetic dataset were all fairly close together. As a result, the novelty adjustment acted as a small bonus rather than significantly changing the recommendations.

Overall, this comparison shows that it is possible to incorporate a user experience goal into the recommendation process while still preserving the model's original recommendations.

### Why This Step Matters

This step matters because it demonstrates that recommendation systems can be evaluated using more than one objective. While prediction accuracy helps determine how well a model estimates user ratings, user experience goals such as novelty can improve the overall recommendation process.

In this project, adding novelty did not dramatically change the recommendations because the movies in the synthetic dataset had similar popularity. However, it demonstrated how recommendation scores can be adjusted to encourage discovery while still preserving the recommendations generated by the original model.

This comparison illustrates that recommendation systems often involve balancing multiple objectives rather than optimizing only for prediction accuracy.

# Conclusion

The goal of this project was to compare two recommendation approaches while also exploring how a user experience goal could be incorporated into the recommendation process. Using the same synthetic movie ratings dataset from my previous projects allowed me to focus on comparing the algorithms under consistent conditions without introducing additional variables from a new dataset.

Overall, the SVD Matrix Factorization model produced a Mean Absolute Error (MAE) of approximately 0.54 on a 1-to-5 rating scale. This indicates that, on average, the predicted ratings differed from the actual user ratings by about half of one rating point. While I also evaluated the Item-Based Collaborative Filtering model, I found that the way it was evaluated in this notebook did not provide a meaningful MAE comparison. Rather than presenting the 0.0 MAE as if it represented perfect performance, I felt it was more appropriate to explain why that result occurred and acknowledge the limitations of the evaluation.

Beyond prediction accuracy, I also explored novelty as a user experience objective. Instead of recommending movies based only on predicted ratings, I adjusted the recommendation scores by giving a small bonus to movies that received fewer ratings. In this synthetic dataset, the novelty scores were very similar because most movies had nearly the same number of ratings. As a result, the recommendation scores increased slightly while the overall ranking of the recommended movies remained unchanged. Although the impact was modest, the exercise demonstrated how recommendation systems can balance prediction accuracy with helping users discover less frequently rated content.

If I were extending this project, one improvement I would make would be to create a train/test split before building either recommendation model. By temporarily hiding a portion of the known ratings, both the Item-Based Collaborative Filtering model and the SVD model could be asked to predict the same hidden ratings. Evaluating both algorithms under identical conditions would provide a more meaningful comparison of their predictive performance.

Another limitation of this project is that it relied entirely on offline evaluation. While offline metrics such as MAE provide valuable information about prediction accuracy, they cannot measure how real users respond to recommendations. In an online evaluation environment, I would measure user interactions such as which recommendations were selected, whether users watched the recommended movies, and whether they continued engaging with the recommendation system over time. I would likely use A/B testing to compare different recommendation strategies and evaluate not only prediction accuracy but also user satisfaction, engagement, and discovery of new content.

Overall, this project reinforced that building an effective recommender system involves more than producing accurate predictions. A successful recommendation system should also consider the overall user experience by helping users discover content that is both relevant and interesting. Working through this project gave me a better understanding of the trade-offs involved in designing recommender systems and showed me that evaluating recommendation quality requires looking beyond a single performance metric.